# NYC Yellow Taxi Trip Data Analysis

## Business Objective

The objective of this project is to analyze NYC Yellow Taxi trip data for January 2025 and identify clear, defensible business and operational insights related to trip demand, distance, duration, passenger patterns, time trends, locations, payment types, vendors, and rate codes.

The analysis focuses on data cleaning, data transformation, feature engineering, aggregation, statistical analysis, visualization, and interpretation using Python.

## Problem Statement

The NYC Yellow Taxi dataset contains trip-level records for January 2025, including pickup and drop-off times, trip distance, passenger count, locations, vendors, payment types, and rate codes.

The goal is to examine the quality and characteristics of the data, identify meaningful patterns in taxi demand and trip behavior, and communicate evidence-based findings through statistical analysis and visualization.

## Dataset Description

- **Dataset:** NYC Yellow Taxi Trip Data
- **Period:** January 2025
- **File:** `yellow_tripdata_2025-01.parquet`
- **Format:** Parquet
- **Analysis level:** Individual taxi trip records
- **Main analysis areas:** Trip activity, distance, duration, passenger patterns, time trends, pickup and drop-off locations, payment types, vendors, and rate codes.

This is a trip-level operational dataset. Location values such as `PULocationID` and `DOLocationID` represent TLC Taxi Zone IDs and are analyzed as IDs rather than assumed neighborhood names.

## Project Workflow

Raw Taxi Data
↓
Understand Dataset
↓
Data Quality Assessment
↓
Data Cleaning
↓
Feature Engineering
↓
Statistical Analysis
↓
Exploratory Data Analysis
↓
Visualization
↓
Business Questions
↓
Insights
↓
Final Conclusion

The analysis includes:

- Understanding the business problem and dataset structure
- Loading and inspecting the Parquet dataset
- Performing data-quality checks
- Identifying missing, duplicate, invalid, and suspicious records
- Cleaning relevant data
- Creating datetime-based features
- Calculating trip duration and average speed
- Performing univariate, bivariate, and multivariate analysis
- Creating and interpreting meaningful visualizations
- Answering business questions using evidence from the dataset
- Summarizing major findings and limitations

## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [ ]:
df = pd.read_parquet("yellow_tripdata_2025-01.parquet")

## Initial Data Exploration

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.nunique()

## Data Quality Assessment

In [ ]:
missing_count = df.isnull().sum()
missing_count

In [ ]:
missing_percentage = (missing_count / df.shape[0]) * 100
missing_percentage

In [ ]:
df.duplicated().sum()

In [ ]:
print("Pickup datetime:")
print(df['tpep_pickup_datetime'].dtype)

print("\nDropoff datetime:")
print(df['tpep_dropoff_datetime'].dtype)

In [ ]:
print("Negative trip distances:", (df['trip_distance'] < 0).sum())
print("Zero trip distances:", (df['trip_distance'] == 0).sum())
print("Maximum trip distance:", df['trip_distance'].max())

In [ ]:
print("Unique passenger counts:")
print(df['passenger_count'].unique())

print("\nPassenger count distribution:")
print(df['passenger_count'].value_counts())

print("\nMissing passenger count:")
print(df['passenger_count'].isnull().sum())

In [ ]:
trip_duration_minutes = (
    df['tpep_dropoff_datetime'] -
    df['tpep_pickup_datetime']
).dt.total_seconds() / 60

(trip_duration_minutes < 0).sum()

In [ ]:
(trip_duration_minutes == 0).sum()

In [ ]:
print("Trips longer than 24 hours:", (trip_duration_minutes > 1440).sum())

df[trip_duration_minutes > 1440][
    ['tpep_pickup_datetime',
     'tpep_dropoff_datetime',
     'trip_distance']
].head()

In [ ]:
print("Trips above 100 miles:", (df['trip_distance'] > 100).sum())
print("Trips above 1000 miles:", (df['trip_distance'] > 1000).sum())

In [ ]:
print("VendorID:")
print(df['VendorID'].unique())

print("\nRatecodeID:")
print(df['RatecodeID'].unique())

print("\nStore and Forward Flag:")
print(df['store_and_fwd_flag'].unique())

print("\nPayment Type:")
print(df['payment_type'].unique())

In [ ]:
print("PULocationID:")
print(df['PULocationID'].min(), df['PULocationID'].max())

print("\nDOLocationID:")
print(df['DOLocationID'].min(), df['DOLocationID'].max())

In [ ]:
print("Suspicious records:")
print("Zero-distance trips:", (df['trip_distance'] == 0).sum())
print("Missing passenger count:", df['passenger_count'].isnull().sum())
print("Drop-off before pickup:", (trip_duration_minutes < 0).sum())
print("Zero-duration trips:", (trip_duration_minutes == 0).sum())
print("Trips longer than 24 hours:", (trip_duration_minutes > 1440).sum())
print("Trips above 1000 miles:", (df['trip_distance'] > 1000).sum())

## Data Cleaning

In [ ]:
cleaned_df = df.copy()

In [ ]:
print("Missing values:")
cleaned_df.isnull().sum()

In [ ]:
cleaned_df.duplicated().sum()

In [ ]:
cleaned_df['tpep_pickup_datetime'] = cleaned_df['tpep_pickup_datetime'].astype('datetime64[us]')
cleaned_df['tpep_dropoff_datetime'] = cleaned_df['tpep_dropoff_datetime'].astype('datetime64[us]')

In [ ]:
print("Passenger count values:")
print(cleaned_df['passenger_count'].unique())

print("\nPassenger count distribution:")
cleaned_df['passenger_count'].value_counts()

In [ ]:
print("Negative trip distances:", (cleaned_df['trip_distance'] < 0).sum())
print("Zero trip distances:", (cleaned_df['trip_distance'] == 0).sum())

In [ ]:
(trip_duration_minutes < 0).sum()

In [ ]:
cleaned_df = cleaned_df[trip_duration_minutes >= 0]

In [ ]:
print("Original rows:", df.shape[0])
print("Rows after cleaning:", cleaned_df.shape[0])
print("Rows removed:", df.shape[0] - cleaned_df.shape[0])

In [ ]:
print("Trips longer than 24 hours:", (trip_duration_minutes > 1440).sum())

### Cleaning Decisions

- A separate cleaned DataFrame was created so that the original dataset remains unchanged.
- Missing values were investigated and retained because they occur in a significant portion of the dataset and should not be replaced without knowing their actual meaning.
- No duplicate records were found, so no duplicate rows were removed.
- The pickup and drop-off datetime columns were kept in appropriate datetime format.
- Unusual passenger counts were investigated but not automatically removed.
- No negative trip distances were found.
- Zero-distance trips were investigated and retained because they are not automatically invalid.
- 124 records with drop-off occurring before pickup were removed because they produced negative trip durations.
- 20 trips longer than 24 hours were identified and retained because they were unusual but not confirmed to be invalid.

## Feature Engineering

In [ ]:
cleaned_df['trip_duration_minutes'] = (
    cleaned_df['tpep_dropoff_datetime'] -
    cleaned_df['tpep_pickup_datetime']
).dt.total_seconds() / 60

cleaned_df['trip_duration_minutes'].head()

In [ ]:
cleaned_df['pickup_date'] = cleaned_df['tpep_pickup_datetime'].dt.date

cleaned_df['pickup_date'].head()

In [ ]:
cleaned_df['pickup_hour'] = cleaned_df['tpep_pickup_datetime'].dt.hour

cleaned_df['pickup_hour'].head()

In [ ]:
cleaned_df['day_of_week'] = cleaned_df['tpep_pickup_datetime'].dt.day_name()

cleaned_df['day_of_week'].head()

In [ ]:
cleaned_df['is_weekend'] = (
    (cleaned_df['day_of_week'] == 'Saturday') |
    (cleaned_df['day_of_week'] == 'Sunday')
)

cleaned_df['is_weekend'].value_counts()

In [ ]:
cleaned_df['average_speed_mph'] = np.nan

cleaned_df.loc[cleaned_df['trip_duration_minutes'] > 0, 'average_speed_mph'] = (
    cleaned_df['trip_distance'] /
    (cleaned_df['trip_duration_minutes'] / 60)
)

cleaned_df['average_speed_mph'].head()

In [ ]:
cleaned_df['distance_category'] = 'Short'

cleaned_df.loc[
    cleaned_df['trip_distance'] >= 2,
    'distance_category'
] = 'Medium'

cleaned_df.loc[
    cleaned_df['trip_distance'] >= 5,
    'distance_category'
] = 'Long'

cleaned_df['distance_category'].value_counts()

In [ ]:
cleaned_df['duration_category'] = 'Very Short'

cleaned_df.loc[
    cleaned_df['trip_duration_minutes'] >= 5,
    'duration_category'
] = 'Short'

cleaned_df.loc[
    cleaned_df['trip_duration_minutes'] >= 10,
    'duration_category'
] = 'Medium'

cleaned_df.loc[
    cleaned_df['trip_duration_minutes'] >= 20,
    'duration_category'
] = 'Long'

cleaned_df['duration_category'].value_counts()

In [ ]:
cleaned_df['route'] = (
    cleaned_df['PULocationID'].astype(str) +
    ' -> ' +
    cleaned_df['DOLocationID'].astype(str)
)

cleaned_df['route'].head()

In [ ]:
cleaned_df[
    ['trip_duration_minutes',
     'pickup_date',
     'pickup_hour',
     'day_of_week',
     'is_weekend',
     'average_speed_mph',
     'distance_category',
     'duration_category',
     'route']
].head()

## Trip Distance & Duration Analysis

In [ ]:
sns.histplot(cleaned_df['trip_distance'], bins=50)
plt.title('Distribution of Trip Distance')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Number of Trips')
plt.show()

The trip distance distribution is concentrated at shorter distances, while a small number of extreme values create a long right tail. Therefore, the median is more representative of a typical trip than the mean.

In [ ]:
print("Mean:", np.mean(cleaned_df['trip_distance']))
print("Median:", np.median(cleaned_df['trip_distance']))
print("Standard deviation:", np.std(cleaned_df['trip_distance']))
print("25th percentile:", np.quantile(cleaned_df['trip_distance'], 0.25))
print("50th percentile:", np.quantile(cleaned_df['trip_distance'], 0.50))
print("75th percentile:", np.quantile(cleaned_df['trip_distance'], 0.75))

In [ ]:
Q1 = np.quantile(cleaned_df['trip_distance'], 0.25)
Q3 = np.quantile(cleaned_df['trip_distance'], 0.75)

IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper limit:", upper_limit)
print("Outliers:", (cleaned_df['trip_distance'] > upper_limit).sum())

In [ ]:
sns.histplot(cleaned_df['trip_duration_minutes'], bins=50)
plt.title('Distribution of Trip Duration')
plt.xlabel('Trip Duration (minutes)')
plt.ylabel('Number of Trips')
plt.show()

The trip duration distribution is concentrated around shorter trips, with a small number of unusually long trips creating a right-skewed distribution. The median therefore gives a better representation of a typical trip duration.

In [ ]:
print("Mean:", np.mean(cleaned_df['trip_duration_minutes']))
print("Median:", np.median(cleaned_df['trip_duration_minutes']))
print("Minimum:", np.min(cleaned_df['trip_duration_minutes']))
print("Maximum:", np.max(cleaned_df['trip_duration_minutes']))

In [ ]:
Q1 = np.quantile(cleaned_df['trip_duration_minutes'], 0.25)
Q3 = np.quantile(cleaned_df['trip_duration_minutes'], 0.75)

IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper limit:", upper_limit)
print("Outliers:", (cleaned_df['trip_duration_minutes'] > upper_limit).sum())

## Passenger Analysis

In [ ]:
sns.countplot(x='passenger_count', data=cleaned_df)
plt.title('Passenger Count Distribution')
plt.xlabel('Passenger Count')
plt.ylabel('Number of Trips')
plt.show()

The passenger count distribution is strongly concentrated at one passenger. Unusual passenger counts and missing passenger-count records are present, so these values were retained and considered during the analysis.

In [ ]:
print("Passenger count distribution:")
print(cleaned_df['passenger_count'].value_counts())

print("\nMissing passenger count:")
print(cleaned_df['passenger_count'].isnull().sum())

## Time-Based Analysis


In [ ]:
hour_counts = cleaned_df['pickup_hour'].value_counts().sort_index()
hour_counts

In [ ]:
sns.barplot(x=hour_counts.index, y=hour_counts.values)
plt.title('Trips by Pickup Hour')
plt.xlabel('Pickup Hour')
plt.ylabel('Number of Trips')
plt.show()

The number of taxi trips varies considerably throughout the day. Trip activity is highest at 6:00 PM, while the lowest activity occurs at 4:00 AM.

In [ ]:
print("Highest trip activity:")
print(hour_counts.sort_values(ascending=False).head(1))

print("\nLowest trip activity:")
print(hour_counts.sort_values().head(1))

In [ ]:
day_counts = cleaned_df['day_of_week'].value_counts()

day_counts

In [ ]:
sns.barplot(
    x=day_counts.index,
    y=day_counts.values
)

plt.title('Trips by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45)
plt.show()

Thursday has the highest number of trips among the days of the week, while Monday has the lowest. Overall, weekday activity is higher than weekend activity.

In [ ]:
payment_counts = cleaned_df['payment_type'].value_counts()

payment_counts

In [ ]:
sns.countplot(x='payment_type', data=cleaned_df)
plt.title('Payment Type Distribution')
plt.xlabel('Payment Type')
plt.ylabel('Number of Trips')
plt.show()

Payment Type 1 is the most frequently recorded payment type, accounting for approximately 70.34% of the cleaned trips. Payment Type 0 is also a significant group and is kept separate because its detailed meaning was not assigned without an official code mapping.

In [ ]:
vendor_counts = cleaned_df['VendorID'].value_counts()

vendor_counts

In [ ]:
sns.countplot(x='VendorID', data=cleaned_df)
plt.title('Trips by Vendor')
plt.xlabel('Vendor ID')
plt.ylabel('Number of Trips')
plt.show()

The trip count shows that the vendors do not contribute an equal number of trips. One vendor accounts for the majority of the recorded trips, while the other vendors represent smaller shares.

In [ ]:
rate_counts = cleaned_df['RatecodeID'].value_counts()

rate_counts

In [ ]:
sns.countplot(x='RatecodeID', data=cleaned_df)
plt.title('Rate Code Distribution')
plt.xlabel('Rate Code ID')
plt.ylabel('Number of Trips')
plt.show()

In [ ]:
hour_counts = cleaned_df['pickup_hour'].value_counts().sort_index()

hour_counts

In [ ]:
print("Highest trip activity:")
print(hour_counts.sort_values(ascending=False).head(1))

print("\nLowest trip activity:")
print(hour_counts.sort_values().head(1))

In [ ]:
day_counts = cleaned_df['day_of_week'].value_counts()

day_counts

In [ ]:
print("Highest trip activity day:")
print(day_counts.sort_values(ascending=False).head(1))

print("\nLowest trip activity day:")
print(day_counts.sort_values().head(1))

In [ ]:
weekend_counts = cleaned_df['is_weekend'].value_counts()

weekend_counts

In [ ]:
sns.barplot(x=weekend_counts.index, y=weekend_counts.values)

plt.title('Weekday vs Weekend Trips')
plt.xlabel('Is Weekend')
plt.ylabel('Number of Trips')
plt.show()

Weekday trips account for approximately 74.71% of the cleaned dataset, while weekend trips account for approximately 25.29%. This shows that taxi activity is more concentrated on weekdays.


In [ ]:
total_trips = np.sum(weekend_counts)

print("Weekday percentage:", (weekend_counts[False] / total_trips) * 100)
print("Weekend percentage:", (weekend_counts[True] / total_trips) * 100)

In [ ]:
hour_day = cleaned_df.pivot_table(
    index='pickup_hour',
    columns='day_of_week',
    values='VendorID',
    aggfunc='count'
)

hour_day = hour_day[
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
     'Friday', 'Saturday', 'Sunday']
]

hour_day

In [ ]:
sns.heatmap(hour_day)

plt.title('Trips by Hour and Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Pickup Hour')
plt.show()

In [ ]:
avg_duration_hour = cleaned_df.groupby('pickup_hour').agg(
    {'trip_duration_minutes': 'mean'}
)

avg_duration_hour

In [ ]:
sns.barplot(
    x=avg_duration_hour.index,
    y=avg_duration_hour['trip_duration_minutes']
)

plt.title('Average Trip Duration by Pickup Hour')
plt.xlabel('Pickup Hour')
plt.ylabel('Average Trip Duration (minutes)')
plt.show()

The average trip duration varies across pickup hours. The longest average trip duration occurs around the afternoon period, while some early-morning hours have shorter average durations.

In [ ]:
avg_duration_hour.sort_values(
    'trip_duration_minutes',
    ascending=False
).head(5)

In [ ]:
avg_distance_hour = cleaned_df.groupby('pickup_hour').agg(
    {'trip_distance': 'mean'}
)

avg_distance_hour

In [ ]:
sns.barplot(
    x=avg_distance_hour.index,
    y=avg_distance_hour['trip_distance']
)

plt.title('Average Trip Distance by Pickup Hour')
plt.xlabel('Pickup Hour')
plt.ylabel('Average Trip Distance (miles)')
plt.show()

Average trip distance varies considerably across pickup hours. The highest average distance occurs during the early-morning hours, which is influenced by a small number of extremely large distance values.

In [ ]:
avg_distance_hour.sort_values(
    'trip_distance',
    ascending=False
).head(5)

In [ ]:
avg_speed_hour = cleaned_df.groupby('pickup_hour').agg(
    {'average_speed_mph': 'mean'}
)

avg_speed_hour

In [ ]:
sns.barplot(
    x=avg_speed_hour.index,
    y=avg_speed_hour['average_speed_mph']
)

plt.title('Average Speed by Pickup Hour')
plt.xlabel('Pickup Hour')
plt.ylabel('Average Speed (mph)')
plt.show()

Average speed varies significantly across pickup hours. Some hours show extremely high average speeds because of unusual distance and duration records, so these values should be interpreted cautiously.

In [ ]:
print("Minimum average speed:")
print(avg_speed_hour.sort_values('average_speed_mph').head(1))

print("\nMaximum average speed:")
print(avg_speed_hour.sort_values(
    'average_speed_mph',
    ascending=False
).head(1))

In [ ]:
print("Average trip distance:", np.mean(cleaned_df['trip_distance']))
print("Median trip distance:", np.median(cleaned_df['trip_distance']))

In [ ]:
print("Average trip duration:", np.mean(cleaned_df['trip_duration_minutes']))
print("Median trip duration:", np.median(cleaned_df['trip_duration_minutes']))

In [ ]:
distance_duration_corr = cleaned_df[
    ['trip_distance', 'trip_duration_minutes']
].corr()

distance_duration_corr

In [ ]:
sns.scatterplot(
    x='trip_distance',
    y='trip_duration_minutes',
    data=cleaned_df
)

plt.title('Trip Distance vs Trip Duration')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Trip Duration (minutes)')
plt.show()

In [ ]:
high_distance_low_duration = cleaned_df[
    (cleaned_df['trip_distance'] > 100) &
    (cleaned_df['trip_duration_minutes'] < 10)
]

high_distance_low_duration.shape

In [ ]:
high_distance_low_duration[
    ['trip_distance', 'trip_duration_minutes']
].head(10)

In [ ]:
low_distance_high_duration = cleaned_df[
    (cleaned_df['trip_distance'] < 1) &
    (cleaned_df['trip_duration_minutes'] > 60)
]

low_distance_high_duration.shape

In [ ]:
low_distance_high_duration[
    ['trip_distance', 'trip_duration_minutes']
].head(10)

In [ ]:
print("High distance and low duration trips:",
      len(high_distance_low_duration))

print("Low distance and high duration trips:",
      len(low_distance_high_duration))

In [ ]:
passenger_counts = cleaned_df['passenger_count'].value_counts()

passenger_counts

In [ ]:
one_passenger = (cleaned_df['passenger_count'] == 1).sum()
total_passenger_trips = cleaned_df['passenger_count'].count()

one_passenger_percentage = (one_passenger / total_passenger_trips) * 100

print("Trips with one passenger:", one_passenger)
print("Percentage of trips with one passenger:", one_passenger_percentage)

In [ ]:
avg_distance_passenger = cleaned_df.groupby('passenger_count')['trip_distance'].mean()

avg_distance_passenger

In [ ]:
avg_duration_passenger = cleaned_df.groupby('passenger_count')['trip_duration_minutes'].mean()

avg_duration_passenger

In [ ]:
passenger_weekend_table = cleaned_df.pivot_table(
    index='passenger_count',
    columns='is_weekend',
    values='trip_distance',
    aggfunc='count'
)

passenger_weekend_table

In [ ]:
sns.countplot(
    x='passenger_count',
    data=cleaned_df
)

plt.title('Passenger Count Distribution')
plt.xlabel('Passenger Count')
plt.ylabel('Number of Trips')
plt.show()

In [ ]:
sns.boxplot(
    x='passenger_count',
    y='trip_distance',
    data=cleaned_df
)

plt.title('Trip Distance by Passenger Count')
plt.xlabel('Passenger Count')
plt.ylabel('Trip Distance (miles)')
plt.show()

The boxplot shows how trip distance varies across passenger counts. Most passenger groups contain shorter trips, while extreme distances appear as outliers.

In [ ]:
sns.barplot(
    x=avg_duration_passenger.index,
    y=avg_duration_passenger.values
)

plt.title('Average Trip Duration by Passenger Count')
plt.xlabel('Passenger Count')
plt.ylabel('Average Trip Duration (minutes)')
plt.show()

Average trip duration varies across passenger-count groups. The results for very small passenger groups should be interpreted carefully because they contain relatively few observations.

In [ ]:
sns.countplot(
    x='passenger_count',
    hue='is_weekend',
    data=cleaned_df
)

plt.title('Passenger Count Distribution: Weekday vs Weekend')
plt.xlabel('Passenger Count')
plt.ylabel('Number of Trips')
plt.show()

Both weekday and weekend trips are dominated by one-passenger trips. The overall passenger pattern is therefore similar across weekdays and weekends.

## Pickup & Drop-off Location Analysis

This section analyzes pickup and drop-off locations using PULocationID and DOLocationID.

The analysis identifies the most frequently used pickup and drop-off locations,
the most common pickup-to-drop-off routes, and the routes with the highest
average trip distance and duration.

In [ ]:
pickup_counts = cleaned_df['PULocationID'].value_counts()

top_pickup_locations = pickup_counts.head(10)

top_pickup_locations

In [ ]:
sns.barplot(
    x=top_pickup_locations.index,
    y=top_pickup_locations.values
)

plt.title('Top 10 Pickup Locations')
plt.xlabel('Pickup Location ID')
plt.ylabel('Number of Trips')
plt.show()

The top 10 pickup locations show where the largest number of taxi trips begin.
The location with the highest count represents the most frequently used pickup
zone in the dataset.

In [ ]:
dropoff_counts = cleaned_df['DOLocationID'].value_counts()

top_dropoff_locations = dropoff_counts.head(10)

top_dropoff_locations

In [ ]:
sns.barplot(
    x=top_dropoff_locations.index,
    y=top_dropoff_locations.values
)

plt.title('Top 10 Drop-off Locations')
plt.xlabel('Drop-off Location ID')
plt.ylabel('Number of Trips')
plt.show()

The top 10 drop-off locations show the zones where the largest number of taxi
trips end. These locations represent the most frequently used drop-off zones.

In [ ]:
route_counts = cleaned_df['route'].value_counts()

top_routes = route_counts.head(10)

top_routes

In [ ]:
route_table = cleaned_df.groupby(
    ['PULocationID', 'DOLocationID']
).agg(
    Number_of_Trips=('trip_distance', 'count')
)

route_table = route_table.sort_values(
    'Number_of_Trips',
    ascending=False
)

route_table.head(10)

The route analysis identifies the pickup-to-drop-off combinations with the
highest number of trips. These routes represent the most frequently travelled
connections in the dataset.

In [ ]:
print("Top 10 Pickup Locations:")
print(top_pickup_locations)

print("\nTop 10 Drop-off Locations:")
print(top_dropoff_locations)

The most common pickup and drop-off locations can be compared using their
Location IDs. Some locations may appear in both lists, indicating that they
are important both as trip origins and destinations. However, the rankings
may differ because a location can generate many pickups without receiving the
same number of drop-offs.

In [ ]:
route_distance = cleaned_df.groupby(
    ['PULocationID', 'DOLocationID']
).agg(
    Average_Distance=('trip_distance', 'mean')
)

route_distance = route_distance.sort_values(
    'Average_Distance',
    ascending=False
)

route_distance.head(10)

The routes with the highest average trip distance represent the pickup-to-
drop-off combinations associated with the longest average reported distances.
Very rare routes may appear near the top because a small number of unusual
trips can strongly affect the average.

In [ ]:
route_duration = cleaned_df.groupby(
    ['PULocationID', 'DOLocationID']
).agg(
    Average_Duration=('trip_duration_minutes', 'mean')
)

route_duration = route_duration.sort_values(
    'Average_Duration',
    ascending=False
)

route_duration.head(10)

The routes with the highest average trip duration represent the pickup-to-
drop-off combinations that take the longest on average. These results should
be interpreted carefully because routes with very few trips or unusual long-
duration records can have a high average.

### Summary

- The top pickup locations identify the zones generating the highest number of trips.
- The top drop-off locations identify the zones receiving the highest number of trips.
- Route analysis identifies the most frequently travelled pickup-to-drop-off combinations.
- Pickup and drop-off rankings can be compared to identify locations that are important on both sides of a trip.
- Route-level average distance and duration help identify longer-distance and longer-duration travel patterns.
- Location IDs are analyzed as TLC Taxi Zone IDs and are not treated as neighborhood names.
- Rare routes and extreme records should be interpreted carefully because they can strongly affect average distance and duration.

## Payment Type Analysis

This section analyzes the distribution of payment types and compares trip
distance, trip duration, and passenger count across payment types.

In [ ]:
payment_counts = cleaned_df['payment_type'].value_counts()

payment_counts

In [ ]:
payment_counts.head(1)

The payment type with the highest number of trips is the most frequently used
payment type in the dataset.

In [ ]:
payment_percentage = (
    payment_counts / payment_counts.sum()
) * 100

payment_percentage

The percentage analysis shows how the total trips are distributed across the
different payment types. The most common payment type represents the largest
share of trips.

In [ ]:
sns.countplot(
    x='payment_type',
    data=cleaned_df
)

plt.title('Payment Type Distribution')
plt.xlabel('Payment Type')
plt.ylabel('Number of Trips')
plt.show()

The count plot shows that payment types are not evenly distributed.
One payment type accounts for the majority of trips, while the remaining
payment types occur less frequently.

In [ ]:
sns.barplot(
    x=payment_percentage.index,
    y=payment_percentage.values
)

plt.title('Percentage of Trips by Payment Type')
plt.xlabel('Payment Type')
plt.ylabel('Percentage of Trips')
plt.show()

The percentage distribution confirms that Payment Type 1 represents the majority of recorded trips, while the remaining payment types account for smaller portions of the dataset.

In [ ]:
avg_distance_payment = cleaned_df.groupby(
    'payment_type'
)['trip_distance'].mean()

avg_distance_payment

In [ ]:
sns.barplot(
    x=avg_distance_payment.index,
    y=avg_distance_payment.values
)

plt.title('Average Trip Distance by Payment Type')
plt.xlabel('Payment Type')
plt.ylabel('Average Trip Distance (miles)')
plt.show()

The average trip distance varies across payment types. The differences should
be interpreted as observed patterns in the dataset and do not by themselves
indicate that one payment type causes longer or shorter trips.

In [ ]:
avg_duration_payment = cleaned_df.groupby(
    'payment_type'
)['trip_duration_minutes'].mean()

avg_duration_payment

In [ ]:
sns.barplot(
    x=avg_duration_payment.index,
    y=avg_duration_payment.values
)

plt.title('Average Trip Duration by Payment Type')
plt.xlabel('Payment Type')
plt.ylabel('Average Trip Duration (minutes)')
plt.show()

The average trip duration differs across payment types. These differences
describe the characteristics of trips associated with each payment type and
should not be interpreted as causal relationships.

In [ ]:
avg_passenger_payment = cleaned_df.groupby(
    'payment_type'
)['passenger_count'].mean()

avg_passenger_payment

The average passenger count can be compared across payment types to identify
whether the typical recorded passenger count differs between payment groups.
Missing passenger counts are not included in the average calculation.

In [ ]:
payment_analysis = cleaned_df.groupby('payment_type').agg(
    Average_Distance=('trip_distance', 'mean'),
    Average_Duration=('trip_duration_minutes', 'mean'),
    Average_Passenger_Count=('passenger_count', 'mean')
)

payment_analysis

### Summary

- Payment type distribution shows which payment methods are most frequently recorded.
- The percentage analysis shows the share of total trips for each payment type.
- Average trip distance varies across payment types.
- Average trip duration varies across payment types.
- Passenger count can also be compared across payment types using the average recorded passenger count.
- These are observed relationships in the dataset and should not be interpreted as causal relationships.

## Vendor Analysis

This section analyzes the distribution of trips across vendors and compares
their trip distance, trip duration, passenger count, and payment type patterns.

In [ ]:
vendor_counts = cleaned_df['VendorID'].value_counts()

vendor_counts

In [ ]:
vendor_percentage = (
    vendor_counts / vendor_counts.sum()
) * 100

vendor_percentage

In [ ]:
sns.countplot(
    x='VendorID',
    data=cleaned_df
)

plt.title('Trip Count by Vendor')
plt.xlabel('Vendor ID')
plt.ylabel('Number of Trips')
plt.show()

The trip count shows that the vendors do not contribute an equal number of
trips. One vendor accounts for the majority of the recorded trips, while the
other vendors represent smaller shares.

In [ ]:
avg_distance_vendor = cleaned_df.groupby(
    'VendorID'
)['trip_distance'].mean()

avg_distance_vendor

In [ ]:
sns.barplot(
    x=avg_distance_vendor.index,
    y=avg_distance_vendor.values
)

plt.title('Average Trip Distance by Vendor')
plt.xlabel('Vendor ID')
plt.ylabel('Average Trip Distance (miles)')
plt.show()

In [ ]:
avg_duration_vendor = cleaned_df.groupby(
    'VendorID'
)['trip_duration_minutes'].mean()

avg_duration_vendor

In [ ]:
sns.barplot(
    x=avg_duration_vendor.index,
    y=avg_duration_vendor.values
)

plt.title('Average Trip Duration by Vendor')
plt.xlabel('Vendor ID')
plt.ylabel('Average Trip Duration (minutes)')
plt.show()

In [ ]:
sns.countplot(
    x='passenger_count',
    hue='VendorID',
    data=cleaned_df
)

plt.title('Passenger Count Distribution by Vendor')
plt.xlabel('Passenger Count')
plt.ylabel('Number of Trips')
plt.show()

Both vendors show a strong concentration of one-passenger trips. The smaller vendor groups contain far fewer observations, so differences in their passenger distributions should be interpreted cautiously.

In [ ]:
vendor_analysis = cleaned_df.groupby('VendorID').agg(
    Number_of_Trips=('trip_distance', 'count'),
    Average_Distance=('trip_distance', 'mean'),
    Average_Duration=('trip_duration_minutes', 'mean'),
    Average_Passenger_Count=('passenger_count', 'mean')
)

vendor_analysis

The vendors show differences in trip volume, average trip distance, and
average trip duration. Passenger-count and payment-type distributions can
also differ across vendors.

These differences describe the observed characteristics of the trips recorded
by each vendor. They should not be interpreted as evidence that one vendor
performs better than another because the dataset does not provide a suitable
overall performance measure.

## Rate Code Analysis

This section analyzes the distribution of RatecodeID and compares trip distance
and trip duration across different rate codes.

In [ ]:
rate_counts = cleaned_df['RatecodeID'].value_counts()

rate_counts

In [ ]:
rate_counts.head(1)

In [ ]:
rate_percentage = (
    rate_counts / rate_counts.sum()
) * 100

rate_percentage

In [ ]:
sns.barplot(
    x=rate_percentage.index,
    y=rate_percentage.values
)

plt.title('Percentage of Trips by Rate Code')
plt.xlabel('Rate Code')
plt.ylabel('Percentage of Trips')
plt.show()

The rate code distribution is highly concentrated in a small number of rate
codes, while some rate codes are used much less frequently.

In [ ]:
avg_distance_rate = cleaned_df.groupby(
    'RatecodeID'
)['trip_distance'].mean()

avg_distance_rate

In [ ]:
sns.barplot(
    x=avg_distance_rate.index,
    y=avg_distance_rate.values
)

plt.title('Average Trip Distance by Rate Code')
plt.xlabel('Rate Code')
plt.ylabel('Average Trip Distance (miles)')
plt.show()

Average trip distance varies across rate codes. Some rate codes are associated
with longer average trips than others.

In [ ]:
avg_duration_rate = cleaned_df.groupby(
    'RatecodeID'
)['trip_duration_minutes'].mean()

avg_duration_rate

In [ ]:
sns.barplot(
    x=avg_duration_rate.index,
    y=avg_duration_rate.values
)

plt.title('Average Trip Duration by Rate Code')
plt.xlabel('Rate Code')
plt.ylabel('Average Trip Duration (minutes)')
plt.show()

Average trip duration differs across rate codes. This indicates that the
typical duration of trips is not the same for every rate code.

In [ ]:
rate_distance_analysis = cleaned_df.groupby('RatecodeID').agg(
    Average_Distance=('trip_distance', 'mean'),
    Maximum_Distance=('trip_distance', 'max')
)

rate_distance_analysis = rate_distance_analysis.sort_values(
    'Maximum_Distance',
    ascending=False
)

rate_distance_analysis

Some rate codes may show unusually high maximum trip distances. These values
should be interpreted carefully because extreme distance records can strongly
affect the results and may represent data anomalies.

In [ ]:
rate_percentage.sort_values().head()

Rate codes with very small percentages are rare in the dataset and may require
further investigation. Rare does not necessarily mean incorrect; the records
should be investigated before making any cleaning decision.

In [ ]:
rate_analysis = cleaned_df.groupby('RatecodeID').agg(
    Number_of_Trips=('trip_distance', 'count'),
    Average_Distance=('trip_distance', 'mean'),
    Average_Duration=('trip_duration_minutes', 'mean'),
    Maximum_Distance=('trip_distance', 'max')
)

rate_analysis

The rate code analysis shows that trip activity is concentrated in a few
commonly used rate codes. Trip distance and duration vary across rate codes,
while some less common rate codes contain unusually high distance records.

Rare rate codes should be treated as records for further investigation rather
than automatically being considered invalid.

## Store & Forward Analysis

This section analyzes the store-and-forward flag and compares trip distance
and trip duration between the different groups.

In [ ]:
store_forward_counts = cleaned_df['store_and_fwd_flag'].value_counts()

store_forward_counts

In [ ]:
store_forward_percentage = (
    store_forward_counts / store_forward_counts.sum()
) * 100

store_forward_percentage

In [ ]:
sns.countplot(
    x='store_and_fwd_flag',
    data=cleaned_df
)

plt.title('Store and Forward Trip Distribution')
plt.xlabel('Store and Forward Flag')
plt.ylabel('Number of Trips')
plt.show()

The majority of recorded store-and-forward trips belong to the N group,
while only a small proportion belong to the Y group. Missing values are
treated separately and are not included in the percentage comparison.

In [ ]:
avg_distance_store_forward = cleaned_df.groupby(
    'store_and_fwd_flag'
)['trip_distance'].mean()

avg_distance_store_forward

In [ ]:
sns.barplot(
    x=avg_distance_store_forward.index,
    y=avg_distance_store_forward.values
)

plt.title('Average Trip Distance by Store and Forward Flag')
plt.xlabel('Store and Forward Flag')
plt.ylabel('Average Trip Distance (miles)')
plt.show()

The average trip distance can be compared between the N and Y groups to
identify differences in their typical trip distances.

In [ ]:
avg_duration_store_forward = cleaned_df.groupby(
    'store_and_fwd_flag'
)['trip_duration_minutes'].mean()

avg_duration_store_forward

In [ ]:
sns.barplot(
    x=avg_duration_store_forward.index,
    y=avg_duration_store_forward.values
)

plt.title('Average Trip Duration by Store and Forward Flag')
plt.xlabel('Store and Forward Flag')
plt.ylabel('Average Trip Duration (minutes)')
plt.show()

The average trip duration can be compared between the N and Y groups to
identify differences in their typical trip duration.

In [ ]:
store_forward_analysis = cleaned_df.groupby('store_and_fwd_flag').agg(
    Number_of_Trips=('trip_distance', 'count'),
    Average_Distance=('trip_distance', 'mean'),
    Average_Duration=('trip_duration_minutes', 'mean')
)

store_forward_analysis

The store-and-forward groups may show differences in trip distance and
duration. These differences describe the characteristics of the recorded
trips in each group.

The analysis does not establish that the store-and-forward status causes
longer or shorter trips. The results should be interpreted as observed
differences in the dataset.

## NumPy Statistical Analysis

This section uses NumPy to calculate statistical measures for important numerical variables.

The analysis includes mean, median, minimum, maximum, variance, standard deviation, and percentiles.

In [ ]:
print("Trip Distance Statistics")

print("Mean:", np.mean(cleaned_df['trip_distance']))
print("Median:", np.median(cleaned_df['trip_distance']))
print("Minimum:", np.min(cleaned_df['trip_distance']))
print("Maximum:", np.max(cleaned_df['trip_distance']))
print("Variance:", np.var(cleaned_df['trip_distance']))
print("Standard deviation:", np.std(cleaned_df['trip_distance']))

print("\nPercentiles:")
print("25th percentile:", np.quantile(cleaned_df['trip_distance'], 0.25))
print("50th percentile:", np.quantile(cleaned_df['trip_distance'], 0.50))
print("75th percentile:", np.quantile(cleaned_df['trip_distance'], 0.75))
print("90th percentile:", np.quantile(cleaned_df['trip_distance'], 0.90))
print("95th percentile:", np.quantile(cleaned_df['trip_distance'], 0.95))
print("99th percentile:", np.quantile(cleaned_df['trip_distance'], 0.99))

In [ ]:
print("Trip Duration Statistics")

print("Mean:", np.mean(cleaned_df['trip_duration_minutes']))
print("Median:", np.median(cleaned_df['trip_duration_minutes']))
print("Minimum:", np.min(cleaned_df['trip_duration_minutes']))
print("Maximum:", np.max(cleaned_df['trip_duration_minutes']))
print("Variance:", np.var(cleaned_df['trip_duration_minutes']))
print("Standard deviation:", np.std(cleaned_df['trip_duration_minutes']))

print("\nPercentiles:")
print("25th percentile:", np.quantile(cleaned_df['trip_duration_minutes'], 0.25))
print("50th percentile:", np.quantile(cleaned_df['trip_duration_minutes'], 0.50))
print("75th percentile:", np.quantile(cleaned_df['trip_duration_minutes'], 0.75))
print("90th percentile:", np.quantile(cleaned_df['trip_duration_minutes'], 0.90))
print("95th percentile:", np.quantile(cleaned_df['trip_duration_minutes'], 0.95))
print("99th percentile:", np.quantile(cleaned_df['trip_duration_minutes'], 0.99))

In [ ]:
passenger_data = cleaned_df['passenger_count'].dropna()

print("Passenger Count Statistics")
print("Mean:", np.mean(passenger_data))
print("Median:", np.median(passenger_data))
print("Minimum:", np.min(passenger_data))
print("Maximum:", np.max(passenger_data))
print("Variance:", np.var(passenger_data))
print("Standard deviation:", np.std(passenger_data))

print("\nPercentiles:")
print("25th percentile:", np.quantile(passenger_data, 0.25))
print("50th percentile:", np.quantile(passenger_data, 0.50))
print("75th percentile:", np.quantile(passenger_data, 0.75))
print("90th percentile:", np.quantile(passenger_data, 0.90))
print("95th percentile:", np.quantile(passenger_data, 0.95))
print("99th percentile:", np.quantile(passenger_data, 0.99))

In [ ]:
speed_data = cleaned_df['average_speed_mph'].dropna()

print("Average Speed Statistics")
print("Mean:", np.mean(speed_data))
print("Median:", np.median(speed_data))
print("Minimum:", np.min(speed_data))
print("Maximum:", np.max(speed_data))
print("Variance:", np.var(speed_data))
print("Standard deviation:", np.std(speed_data))

print("\nPercentiles:")
print("25th percentile:", np.quantile(speed_data, 0.25))
print("50th percentile:", np.quantile(speed_data, 0.50))
print("75th percentile:", np.quantile(speed_data, 0.75))
print("90th percentile:", np.quantile(speed_data, 0.90))
print("95th percentile:", np.quantile(speed_data, 0.95))
print("99th percentile:", np.quantile(speed_data, 0.99))

### Interpretation

The NumPy statistics provide a detailed summary of the central tendency and spread of the main numerical variables.

The difference between the mean and median helps identify the effect of extreme values. The higher percentiles show how the upper end of the distributions behaves, while variance and standard deviation indicate how widely the values are spread.

The average speed variable should be interpreted carefully because extreme trip-distance and trip-duration records can produce unusually high speed values.

## IQR & Outlier Analysis

This section uses the Interquartile Range (IQR) method to identify potential
outliers in trip distance, trip duration, and average speed.

Outliers are investigated before making any decision about removing them.

In [ ]:
Q1_distance = np.quantile(cleaned_df['trip_distance'], 0.25)
Q3_distance = np.quantile(cleaned_df['trip_distance'], 0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

distance_outliers = (
    (cleaned_df['trip_distance'] < lower_distance) |
    (cleaned_df['trip_distance'] > upper_distance)
)

print("Trip Distance")
print("Q1:", Q1_distance)
print("Q3:", Q3_distance)
print("IQR:", IQR_distance)
print("Lower Bound:", lower_distance)
print("Upper Bound:", upper_distance)
print("Number of potential outliers:", distance_outliers.sum())

In [ ]:
Q1_duration = np.quantile(cleaned_df['trip_duration_minutes'], 0.25)
Q3_duration = np.quantile(cleaned_df['trip_duration_minutes'], 0.75)

IQR_duration = Q3_duration - Q1_duration

lower_duration = Q1_duration - 1.5 * IQR_duration
upper_duration = Q3_duration + 1.5 * IQR_duration

duration_outliers = (
    (cleaned_df['trip_duration_minutes'] < lower_duration) |
    (cleaned_df['trip_duration_minutes'] > upper_duration)
)

print("Trip Duration")
print("Q1:", Q1_duration)
print("Q3:", Q3_duration)
print("IQR:", IQR_duration)
print("Lower Bound:", lower_duration)
print("Upper Bound:", upper_duration)
print("Number of potential outliers:", duration_outliers.sum())

In [ ]:
speed_data = cleaned_df['average_speed_mph'].dropna()

Q1_speed = np.quantile(speed_data, 0.25)
Q3_speed = np.quantile(speed_data, 0.75)

IQR_speed = Q3_speed - Q1_speed

lower_speed = Q1_speed - 1.5 * IQR_speed
upper_speed = Q3_speed + 1.5 * IQR_speed

speed_outliers = (
    (cleaned_df['average_speed_mph'] < lower_speed) |
    (cleaned_df['average_speed_mph'] > upper_speed)
)

print("Average Speed")
print("Q1:", Q1_speed)
print("Q3:", Q3_speed)
print("IQR:", IQR_speed)
print("Lower Bound:", lower_speed)
print("Upper Bound:", upper_speed)
print("Number of potential outliers:", speed_outliers.sum())

In [ ]:
print("Extreme Trip Distance Records:")
cleaned_df.sort_values(
    'trip_distance',
    ascending=False
)[['trip_distance', 'trip_duration_minutes', 'average_speed_mph']].head(10)

In [ ]:
print("Extreme Trip Duration Records:")
cleaned_df.sort_values(
    'trip_duration_minutes',
    ascending=False
)[['trip_distance', 'trip_duration_minutes', 'average_speed_mph']].head(10)

In [ ]:
print("Extreme Average Speed Records:")
cleaned_df.sort_values(
    'average_speed_mph',
    ascending=False
)[['trip_distance', 'trip_duration_minutes', 'average_speed_mph']].head(10)

### Outlier Decision

The IQR method identifies a large number of potential outliers in trip distance,
trip duration, and average speed.

These observations are not automatically removed. Some unusual trips may be
valid, while extremely large distance or speed values may indicate data-quality
issues.

The extreme records are therefore retained in the cleaned dataset and treated
as observations requiring further investigation. This preserves the original
information while preventing unusual values from being silently deleted.

Average speed requires additional caution because it is calculated from trip
distance and trip duration. Extremely short durations combined with unusually
large distances can produce unrealistic speed values.

In [ ]:
sns.boxplot(y=cleaned_df['trip_distance'])
plt.title('Trip Distance Outliers')
plt.ylabel('Trip Distance (miles)')
plt.show()

The boxplot shows that most trip distances are concentrated at lower values, with several extreme observations appearing as outliers. These extreme values were retained for investigation rather than automatically removed.

In [ ]:
sns.boxplot(y=cleaned_df['trip_duration_minutes'])
plt.title('Trip Duration Outliers')
plt.ylabel('Trip Duration (minutes)')
plt.show()

Most trip durations are concentrated within a relatively short range, while a small number of very long trips appear as outliers. These observations can strongly affect average duration and therefore require careful interpretation.

In [ ]:
sns.boxplot(y=cleaned_df['average_speed_mph'])
plt.title('Average Speed Outliers')
plt.ylabel('Average Speed (mph)')
plt.show()

The average speed analysis contains several extreme values. These unusually high speeds are likely related to combinations of very large trip distances and very short durations, so the results should be interpreted cautiously.

## Correlation Analysis

This section analyzes the correlation between important numerical variables.

Correlation is used to understand the strength and direction of linear relationships
between variables. A correlation value close to 1 indicates a strong positive
relationship, a value close to -1 indicates a strong negative relationship, and a
value close to 0 indicates a weak or no linear relationship.

In [ ]:
correlation_data = cleaned_df[
    ['passenger_count',
     'trip_distance',
     'trip_duration_minutes',
     'average_speed_mph']
]

correlation_matrix = correlation_data.corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation Matrix')
plt.show()

### Interpretation

The correlation matrix shows the strength and direction of the linear relationships
between passenger count, trip distance, trip duration, and average speed.

The strongest relationships should be identified from the correlation values in the
matrix. Values close to zero indicate weak linear relationships.

Trip distance and trip duration should be interpreted using their actual correlation
value rather than assuming that longer trips always take longer.

In [ ]:
print("Trip distance vs trip duration:",
      correlation_matrix.loc['trip_distance', 'trip_duration_minutes'])

print("Passenger count vs trip distance:",
      correlation_matrix.loc['passenger_count', 'trip_distance'])

### Correlation Findings

**Strong relationships:**  
The strongest relationship can be identified from the correlation matrix by looking
for correlation values closest to 1 or -1.

**Weak relationships:**  
Relationships with correlation values close to 0 indicate weak linear relationships.

**Trip distance and trip duration:**  
The correlation value between trip distance and trip duration is approximately
0.003. This indicates an almost zero linear relationship in the full dataset.

**Passenger count and trip distance:**  
The correlation value should be checked from the correlation matrix. It is not
expected to indicate a strong relationship.

**Correlation and causation:**  
Correlation does not imply causation. A correlation only describes the strength
and direction of a relationship between variables. It does not prove that a change
in one variable causes a change in another variable.

In [ ]:
sns.scatterplot(
    x='trip_distance',
    y='average_speed_mph',
    data=cleaned_df
)

plt.title('Trip Distance vs Average Speed')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Average Speed (mph)')
plt.show()

### Interpretation

The scatter plot shows the relationship between trip distance and average speed.

Most observations are concentrated in the lower distance and speed ranges, while
some extreme observations appear far from the main group. These extreme values are
important because unusually large distances or very short durations can produce
unusually high calculated speeds.

Therefore, the extreme speed values should be treated as potential outliers and
investigated rather than automatically removed.

## Advanced Analysis

This section performs additional analysis to identify demand patterns,
route concentration, speed-related anomalies, distance patterns, and
peak versus off-peak activity.

In [ ]:
day_hour_table = cleaned_df.pivot_table(
    index='day_of_week',
    columns='pickup_hour',
    values='trip_distance',
    aggfunc='count'
)

day_hour_table

In [ ]:
plt.figure(figsize=(12, 6))

sns.heatmap(
    day_hour_table,
    cmap='YlGnBu'
)

plt.title('Taxi Demand by Day and Hour')
plt.xlabel('Pickup Hour')
plt.ylabel('Day of Week')
plt.show()

The heatmap shows that taxi demand changes across both day and hour. Higher activity is concentrated during busy daytime and evening periods, while early-morning hours generally show lower activity.


### Insight

Taxi demand varies considerably across both day and hour. Evening hours show
higher activity on several days, while the early morning hours generally have
lower trip activity.

In [ ]:
top_20_routes = cleaned_df['route'].value_counts().head(20)

top_20_routes

In [ ]:
plt.figure(figsize=(10, 7))

sns.barplot(
    x=top_20_routes.values,
    y=top_20_routes.index
)

plt.title('Top 20 Most Frequently Travelled Routes')
plt.xlabel('Number of Trips')
plt.ylabel('Route')
plt.show()

### Insight

A relatively small number of routes account for a noticeable amount of trip
activity. These frequently travelled routes represent the most common pickup
and drop-off combinations in the dataset.

In [ ]:
route_duration_analysis = cleaned_df.groupby(
    ['PULocationID', 'DOLocationID']
).agg(
    Number_of_Trips=('trip_distance', 'count'),
    Average_Duration=('trip_duration_minutes', 'mean')
)

route_duration_analysis = route_duration_analysis[
    route_duration_analysis['Number_of_Trips'] >= 100
]

route_duration_analysis = route_duration_analysis.sort_values(
    'Average_Duration',
    ascending=False
)

route_duration_analysis.head(10)

### Insight

After applying a minimum trip-count threshold, the routes with the highest
average durations are less likely to be influenced by only a few unusual trips.
These routes can be investigated further for longer travel times.

In [ ]:
speed_analysis = cleaned_df[
    ['trip_distance', 'trip_duration_minutes', 'average_speed_mph']
].sort_values(
    'average_speed_mph',
    ascending=False
)

speed_analysis.head(10)

In [ ]:
print("Minimum average speed:", np.min(cleaned_df['average_speed_mph'].dropna()))
print("Maximum average speed:", np.max(cleaned_df['average_speed_mph'].dropna()))

### Insight

Very high average-speed values are present in the dataset. These values can
result from unusually large trip distances combined with very short recorded
durations.

Therefore, extreme speed values may be related to unusual distances,
very short trips, incorrect timestamps, or other data-quality issues. They are
treated as records requiring further investigation rather than being
automatically removed.

In [ ]:
distance_band_analysis = cleaned_df.groupby(
    'distance_category'
).agg(
    Trip_Count=('trip_distance', 'count'),
    Average_Duration=('trip_duration_minutes', 'mean'),
    Average_Speed=('average_speed_mph', 'mean'),
    Average_Passenger_Count=('passenger_count', 'mean')
)

distance_band_analysis

In [ ]:
sns.barplot(
    x=distance_band_analysis.index,
    y=distance_band_analysis['Trip_Count']
)

plt.title('Trip Count by Distance Category')
plt.xlabel('Distance Category')
plt.ylabel('Number of Trips')
plt.show()

### Insight

Short-distance trips make up the largest share of the recorded trips, while
long-distance trips occur less frequently. The average duration, speed, and
passenger count can be compared across these distance categories using the
summary table.

### Peak and Off-Peak Definition

For this analysis, peak hours are defined as 7 AM to 9 AM and 4 PM to 7 PM,
representing common morning and evening high-demand periods.

All remaining hours are classified as off-peak.

In [ ]:
cleaned_df['time_period'] = 'Off-Peak'

cleaned_df.loc[
    ((cleaned_df['pickup_hour'] >= 7) & (cleaned_df['pickup_hour'] <= 9)) |
    ((cleaned_df['pickup_hour'] >= 16) & (cleaned_df['pickup_hour'] <= 19)),
    'time_period'
] = 'Peak'

In [ ]:
peak_analysis = cleaned_df.groupby(
    'time_period'
).agg(
    Trip_Count=('trip_distance', 'count'),
    Average_Distance=('trip_distance', 'mean'),
    Average_Duration=('trip_duration_minutes', 'mean'),
    Average_Speed=('average_speed_mph', 'mean')
)

peak_analysis

In [ ]:
sns.barplot(
    x=peak_analysis.index,
    y=peak_analysis['Trip_Count']
)

plt.title('Peak vs Off-Peak Trip Count')
plt.xlabel('Time Period')
plt.ylabel('Number of Trips')
plt.show()

### Insight

The peak and off-peak groups show how taxi activity and trip characteristics
differ across the defined time periods. The comparison should be interpreted
using the calculated trip count, average distance, duration, and speed rather
than assuming that peak periods always have higher values for every metric.

In [ ]:
total_trips = cleaned_df.shape[0]

top_20_route_trips = top_20_routes.sum()

top_20_route_percentage = (
    top_20_route_trips / total_trips
) * 100

print("Trips in top 20 routes:", top_20_route_trips)
print("Total trips:", total_trips)
print("Percentage of trips in top 20 routes:", top_20_route_percentage)

### Insight

The percentage calculated above shows how concentrated taxi activity is among
the 20 most frequently travelled routes. A relatively small percentage would
indicate that demand is distributed across many different routes, while a
larger percentage would indicate stronger route concentration.

## Business Questions — Final Executive Summary

### When is taxi demand highest?

Taxi demand is highest at **6:00 PM**, with approximately **267,949 trips** recorded. This indicates that evening hours are one of the busiest periods in the dataset.

### When is taxi demand lowest?

Taxi demand is lowest at **4:00 AM**, with approximately **20,031 trips**. Early-morning hours show substantially lower taxi activity compared with the rest of the day.

### How does weekday demand compare with weekend demand?

Weekday trips account for approximately **74.71%** of the cleaned dataset, while weekend trips account for approximately **25.29%**. Therefore, taxi activity is considerably higher on weekdays.

### What does a typical taxi trip look like in terms of distance and duration?

A typical trip has a **median distance of 1.67 miles** and a **median duration of 11.7 minutes**. The mean distance is approximately **5.86 miles** and the mean duration is approximately **15.03 minutes**. The higher means are influenced by extreme observations.

### What are the most common pickup locations?

The most common pickup location is **Taxi Zone 161**, with approximately **169,977 trips**.

### What are the most common drop-off locations?

The most common drop-off location is **Taxi Zone 236**, with approximately **161,376 trips**.

### What are the most frequently travelled routes?

The most frequently travelled route is **Taxi Zone 237 → Taxi Zone 236**, with approximately **24,839 trips**.

The top 20 routes together account for approximately **6.37% of all cleaned trips**, indicating that taxi activity is distributed across many different routes.

### Are longer trips generally associated with longer durations?

The correlation between trip distance and trip duration is approximately **0.003**, which indicates almost no linear relationship in the full dataset.

The analysis also identified **27 trips with distance above 100 miles and duration below 10 minutes**, and **720 trips with distance below 1 mile and duration above 60 minutes**. These records indicate potential data-quality issues and extreme observations.

### Which hours show unusually long or short trip durations?

The highest average trip duration occurs at **4:00 PM**, at approximately **16.74 minutes**. The shortest average duration occurs at **3:00 AM**, at approximately **12.50 minutes**.

### What passenger-count patterns can be observed?

Trips with **one passenger** are the most common, accounting for approximately **79% of trips with a recorded passenger count**.

Most weekday and weekend trips are also dominated by one-passenger trips. Unusual passenger counts and missing passenger-count values were retained because there was not enough evidence to classify them as invalid.

### Which payment types are most common?

**Payment Type 1** is the most frequently recorded payment type, representing approximately **70.34% of cleaned trips**.

Payment Type 0 is also a significant group. It was kept separate because its detailed meaning was not assigned without an official code mapping.

### How do trip characteristics vary across vendors?

The vendors differ in trip volume, average distance, average duration, passenger-count distribution, and payment-type distribution.

One vendor accounts for the majority of recorded trips, while the other vendor groups contain much smaller numbers of observations. These differences describe observed trip characteristics and do not indicate that one vendor performs better.

### Which rate codes are most frequently used?

**Rate Code 1** is the most frequently recorded non-missing rate code, accounting for approximately **93.92% of non-missing RatecodeID records**.

The remaining rate codes occur much less frequently, and rare rate codes should be interpreted carefully because their sample sizes are small.

### What unusual or potentially problematic records exist?

Several unusual records were identified, including:

- Extremely large trip distances
- Very short durations combined with extremely large distances
- Very long trip durations
- Extremely high calculated speeds
- Zero-distance trips
- Zero-duration trips
- Unusual passenger counts
- Missing values across several columns

For example, some records contain distances of tens of thousands of miles combined with only a few minutes of duration. These observations were retained for investigation rather than automatically deleted.

### What are the most important data-quality limitations?

The main limitations include missing values, extreme trip distances and durations, unusual calculated speeds, zero-distance and zero-duration trips, unusual passenger counts, and rare category values.

The location fields contain Taxi Zone IDs rather than neighborhood names, so a separate lookup table would be required for detailed geographic interpretation.

The dataset also does not provide sufficient information to analyze customer-level behavior, driver performance, vehicle-level performance, profit, or detailed revenue generation.

### What operational questions could a taxi company or transportation analyst investigate next?

Further analysis could investigate:

- How taxi availability should be planned during peak hours.
- Which pickup and drop-off zones require greater taxi availability.
- How demand changes between weekdays and weekends.
- Which routes consistently experience longer trip durations.
- Whether traffic or weather conditions affect trip duration and speed.
- Why certain records contain extreme distances, durations, or speeds.
- How passenger and payment patterns change throughout the day.
- Whether demand is concentrated enough in specific zones or routes to support targeted vehicle allocation.

## Final KPI Summary

The table below summarizes the key metrics obtained from the cleaned NYC Yellow Taxi dataset.

In [ ]:
kpi_summary = {
    'KPI': [
        'Total Trips',
        'Average Trip Distance',
        'Median Trip Distance',
        'Average Trip Duration',
        'Median Trip Duration',
        'Maximum Trip Distance',
        'Maximum Trip Duration',
        'Average Speed',
        'Most Common Passenger Count',
        'Busiest Pickup Hour',
        'Busiest Day',
        'Most Common Payment Type',
        'Most Common Pickup Location',
        'Most Common Drop-off Location',
        'Most Common Route'
    ],

    'Value': [
        cleaned_df.shape[0],
        np.mean(cleaned_df['trip_distance']),
        np.median(cleaned_df['trip_distance']),
        np.mean(cleaned_df['trip_duration_minutes']),
        np.median(cleaned_df['trip_duration_minutes']),
        np.max(cleaned_df['trip_distance']),
        np.max(cleaned_df['trip_duration_minutes']),
        np.mean(cleaned_df['average_speed_mph'].dropna()),
        cleaned_df['passenger_count'].value_counts().head(1).index[0],
        cleaned_df['pickup_hour'].value_counts().sort_values(ascending=False).head(1).index[0],
        cleaned_df['day_of_week'].value_counts().head(1).index[0],
        cleaned_df['payment_type'].value_counts().head(1).index[0],
        cleaned_df['PULocationID'].value_counts().head(1).index[0],
        cleaned_df['DOLocationID'].value_counts().head(1).index[0],
        cleaned_df['route'].value_counts().head(1).index[0]
    ]
}

kpi_summary = pd.DataFrame(kpi_summary)

kpi_summary

## Key Findings

- The cleaned dataset contains **3,475,102 taxi trips** after removing 124 records where the drop-off time occurred before the pickup time.

- Taxi demand is highest at **6:00 PM**, with approximately **267,949 trips**, while the lowest activity occurs at **4:00 AM**, with approximately **20,031 trips**.

- **Weekday trips account for approximately 74.71%** of the cleaned dataset, while weekend trips account for approximately **25.29%**.

- A typical trip is relatively short. The **median trip distance is 1.67 miles** and the **median trip duration is 11.7 minutes**.

- The mean trip distance is approximately **5.86 miles** and the mean trip duration is approximately **15.03 minutes**. The higher mean values are influenced by extreme observations.

- **One-passenger trips are the most common**, accounting for approximately **79% of trips with a recorded passenger count**.

- **Payment Type 1 is the most frequently recorded payment type**, representing approximately **70.34% of cleaned trips**.

- **Taxi Zone 161 is the most common pickup location**, with approximately **169,977 trips**.

- **Taxi Zone 236 is the most common drop-off location**, with approximately **161,376 trips**.

- The most frequently travelled route is **Taxi Zone 237 → Taxi Zone 236**, with approximately **24,839 trips**.

- The correlation between trip distance and trip duration is approximately **0.003**, indicating almost no linear relationship in the full dataset.

- The analysis identified unusual records involving extremely large distances, unusually short or long durations, and extremely high calculated speeds.

- Missing values, extreme observations, zero-distance trips, zero-duration trips, and unusual passenger counts are important data-quality considerations when interpreting the results.

Overall, the analysis shows that taxi demand is concentrated during particular hours and weekdays, most trips are short and involve one passenger, and the dataset contains several data-quality issues that should be considered when interpreting the results.

## Business/Operational Insights

- Taxi demand is highest during the evening, particularly around **6:00 PM**. This suggests that taxi availability should be planned carefully during high-demand periods.

- Weekdays account for approximately **74.71% of trips**, indicating that weekday travel contributes more to overall taxi activity than weekends.

- Since **one-passenger trips are the most common**, taxi demand is largely concentrated around individual passenger trips rather than larger passenger groups.

- The most frequently used pickup and drop-off Taxi Zones can help identify areas with consistently high taxi activity. These locations can be considered when planning taxi availability and operational resources.

- The most frequently travelled route, **Taxi Zone 237 → Taxi Zone 236**, indicates a recurring zone-to-zone travel pattern. Frequently travelled routes can be useful for understanding repeated demand patterns.

- Taxi activity varies considerably by hour and day. This information can support planning of taxi availability during busy and less busy periods.

- The analysis identified extreme trip distances, durations, and calculated speeds. These records should be investigated further because they can affect averages and other statistical measures.

- The difference between mean and median trip distance and duration shows that extreme observations have a noticeable effect on the distributions. Therefore, operational decisions should not rely only on mean values.

- The analysis provides useful information about taxi trip patterns, but additional information such as traffic conditions, weather, vehicle availability, and detailed geographic information would be required for more detailed operational planning.

## Limitations

- The dataset contains missing values in several columns, including `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee`.

- Several extreme values were identified in trip distance, trip duration, and average speed. These values can strongly affect mean-based statistics and should be investigated further.

- Some trips have zero distance or zero duration. These records were retained because they cannot automatically be considered invalid without additional context.

- The dataset contains unusual passenger-count values and missing passenger counts. These values were retained rather than removed without sufficient evidence.

- `PULocationID` and `DOLocationID` represent Taxi Zone IDs. A separate lookup table would be required to describe these locations using neighborhood names.

- The dataset does not provide sufficient information to analyze customer-level behavior, driver performance, vehicle-level performance, profit, or detailed revenue generation.

- Payment types and other coded fields were analyzed using their recorded values. Detailed descriptions of individual codes would require the appropriate official data dictionary.

- The analysis is based on January 2025 data, so the findings may not represent taxi demand patterns during other months or seasons.

- Correlation analysis identifies relationships between numerical variables but does not establish causation.

- Very small groups, such as rare vendor or rate-code categories, should be interpreted carefully because their results may not be representative.

- Additional information such as traffic conditions, weather, vehicle availability, and detailed geographic information could provide further context for operational analysis.

## Final Conclusion

The NYC Yellow Taxi Trip Data Analysis provided an end-to-end exploratory data analysis of January 2025 taxi trip records. The analysis covered data understanding, data-quality assessment, cleaning, feature engineering, statistical analysis, exploratory analysis, visualization, and business-focused interpretation.

After cleaning, the dataset contained **3,475,102 trips**. The analysis found that taxi demand was highest during the evening, with **6:00 PM** recording the highest trip activity. Weekday trips represented approximately **74.71%** of the cleaned dataset, while weekend trips represented approximately **25.29%**.

Most trips were relatively short, with a median trip distance of **1.67 miles** and a median duration of **11.7 minutes**. One-passenger trips were the most common passenger category, and Payment Type 1 was the most frequently recorded payment type.

The analysis also identified important data-quality issues, including missing values, zero-distance trips, zero-duration trips, extreme trip distances, unusually long durations, and extreme calculated speeds. These observations were investigated and retained where there was not enough evidence to classify them as invalid.

The project demonstrates how Python, Pandas, NumPy, Matplotlib, and Seaborn can be used to transform raw transportation data into meaningful statistical and business insights. It also demonstrates the importance of using median, percentiles, outlier analysis, and data-quality checks rather than relying only on simple averages.

Overall, the analysis provides a clear understanding of taxi trip patterns and highlights areas that could be investigated further using additional information such as traffic, weather, vehicle availability, and detailed geographic data.